# Fine-tuning YOLO11s-cls no domínio REAL (fotos do celular)

**Continua a sessão do `train_yolo.ipynb`** — assume que já existem em memória:
`model` (treinado no Roboflow), `CLASS_NAMES`, `SHORT`, `short_names`, `crop_single_grain`,
`idx_of`, `api`, `HF_TOKEN`, `CORR_REPO`, `np`, `plt`, `sns`, sklearn.

Se abriu num runtime novo, rode antes as células 1-7b do `train_yolo.ipynb`.

### Aviso honesto
São ~57 fotos (15/11/19/8/4 por classe). Isso é **pouco** — espere overfitting.
Este notebook é um **probe** pra ver a tendência, não a versão de produção.
O domain shift é de aparência (fundo cinza/luz), que vive no extrator de features —
por isso treinamos um pouco além da cabeça, mas sem descongelar tudo (senão decora as 57).

In [ ]:
# FT-1 — Montar dataset de fine-tuning com as fotos REAIS recortadas (train/val)
import os, shutil, random, tempfile, pathlib
from huggingface_hub import hf_hub_download

REAL_DST = '/content/soja_real'
if os.path.exists(REAL_DST):
    shutil.rmtree(REAL_DST)
rng = random.Random(42)
IMG_EXTS_SET = {'.jpg', '.jpeg', '.png'}
tmpd = tempfile.mkdtemp(prefix='ft_')

all_files = list(api.list_repo_files(CORR_REPO, repo_type='dataset'))
summary = {}
for cls in CLASS_NAMES:
    fs = sorted(f for f in all_files
                if f.startswith(f'{cls}/') and pathlib.Path(f).suffix.lower() in IMG_EXTS_SET)
    rng.shuffle(fs)
    # split estratificado 80/20 (garante >=1 no val se a classe tiver >=2 fotos)
    n_val = max(1, round(len(fs) * 0.2)) if len(fs) >= 2 else 0
    val_set = set(fs[:n_val])
    for f in fs:
        local = hf_hub_download(CORR_REPO, f, repo_type='dataset', token=HF_TOKEN, local_dir=tmpd)
        arr  = np.array(Image.open(local).convert('RGB'))
        crop = crop_single_grain(arr)
        split   = 'val' if f in val_set else 'train'
        outdir  = os.path.join(REAL_DST, split, cls)
        os.makedirs(outdir, exist_ok=True)
        Image.fromarray(crop).save(os.path.join(outdir, pathlib.Path(f).stem + '.jpg'), quality=95)
    summary[SHORT[cls]] = (len(fs) - n_val, n_val)

print('classe -> (train, val)')
for k, v in summary.items():
    print(f'  {k:>13}: {v}')
print('\n⚠️  spotted/skin-damaged têm pouquíssimas fotos — o val por classe será ruidoso.')
print('   Leia o resultado como TENDÊNCIA, não como métrica precisa.')

In [ ]:
# FT-2 — Baseline: o modelo do Roboflow (ANTES do fine-tuning) no val real
import glob

def eval_on_split(yolo_model, split):
    paths, true = [], []
    for cls in CLASS_NAMES:
        for p in sorted(glob.glob(os.path.join(REAL_DST, split, cls, '*.jpg'))):
            paths.append(p); true.append(idx_of[cls])
    if not paths:
        return None, None, None
    imgs = [Image.open(p).convert('RGB') for p in paths]
    pred = [idx_of[yolo_model.names[int(r.probs.top1)]]
            for r in yolo_model.predict(imgs, imgsz=224, verbose=False)]
    true, pred = np.array(true), np.array(pred)
    return (true == pred).mean(), true, pred

base_val_acc, _, _ = eval_on_split(model, 'val')
base_tr_acc,  _, _ = eval_on_split(model, 'train')
print(f'BASELINE (Roboflow, sem fine-tuning) no domínio real:')
print(f'  train real: {base_tr_acc:.1%}')
print(f'  val real:   {base_val_acc:.1%}')

In [ ]:
# FT-3 — Fine-tuning: parte do modelo Roboflow, congela o backbone, LR baixo
#
# FREEZE controla quanto descongela:
#   freeze=10 -> treina SÓ a cabeça (Classify). Mais seguro, mas mexe pouco no domain shift.
#   freeze=9  -> treina o último bloco + cabeça. Adapta mais features (recomendado p/ fundo/luz).
#   freeze=8  -> mais capacidade, mais risco de decorar as 57 fotos.
FREEZE = 9

ft = YOLO(str(model.trainer.best))   # pesos do treino Roboflow como ponto de partida
ft_results = ft.train(
    data=REAL_DST,
    epochs=60,
    imgsz=224,
    batch=16,
    freeze=FREEZE,
    lr0=0.0005,         # 20x menor que o treino base: não destruir o aprendido
    patience=15,
    seed=42,
    project='/content/runs_soja',
    name='yolo11s_finetune',
    exist_ok=True,
    verbose=True,
)
print('\nFine-tuning concluído. save_dir:', ft.trainer.save_dir)

In [ ]:
# FT-4 — Antes x Depois no domínio real + matriz de confusão
from sklearn.metrics import classification_report, confusion_matrix

new_tr_acc,  _, _              = eval_on_split(ft, 'train')
new_val_acc, val_true, val_pred = eval_on_split(ft, 'val')

print('=== Domínio real: ANTES (Roboflow) x DEPOIS (fine-tuned) ===')
print(f'  train: {base_tr_acc:.1%}  ->  {new_tr_acc:.1%}')
print(f'  val:   {base_val_acc:.1%}  ->  {new_val_acc:.1%}')
gap = new_tr_acc - new_val_acc
print(f'\n  gap train-val depois: {gap:.1%}  ', end='')
print('(gap grande = overfitting, esperado com 57 fotos)' if gap > 0.2 else '(gap saudável)')

print('\n=== Relatório no val real (fine-tuned) ===')
print(classification_report(val_true, val_pred, target_names=short_names, zero_division=0))

cm = confusion_matrix(val_true, val_pred, labels=list(range(len(CLASS_NAMES))))
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=short_names, yticklabels=short_names,
            cmap='Greens', linewidths=0.5)
plt.xlabel('Predito'); plt.ylabel('Real')
plt.title(f'Fine-tuned no domínio real — val {new_val_acc:.0%}')
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout(); plt.show()

In [ ]:
# FT-5 (OPCIONAL) — Salvar o modelo fine-tuned no Drive (só se o val real melhorou)
from pathlib import Path
best_ft = Path(ft.trainer.best)
if new_val_acc is not None and new_val_acc > (base_val_acc or 0):
    dest = f'{SAVE_DIR}/soja_yolo11s_finetuned.pt'
    shutil.copy(best_ft, dest)
    print(f'✅ val real melhorou ({base_val_acc:.1%} -> {new_val_acc:.1%}) — salvo em {dest}')
else:
    print(f'❌ val real NÃO melhorou ({base_val_acc:.1%} -> {new_val_acc:.1%}) — não salvei.')
    print('   Provável: 57 fotos é pouco pra esse domain shift. Colete mais ou padronize a captura.')